In [ ]:
from pathlib import Path
import torch

CKPT_DIR = Path('D:\\Tree-Structured Parzen Estimator\\main\\[6] model')

CHECKPOINTS = [
    'densenet121_best.pth',
    'inception_v3_best.pth',
    'xception_best.pth',
    'vit_best.pth',
    'ensemble_best.pth',
]


def inspect_single_model(ckpt: dict, path: Path):
    """Checkpoint model tunggal (DenseNet/InceptionV3/Xception/ViT)."""
    state_dict = ckpt.get('state_dict', {})
    print(f"Jumlah tensor     : {len(state_dict)}")
    total_params = sum(v.numel() for v in state_dict.values())
    print(f"Total parameter   : {total_params / 1e6:.2f} M")

    print(f"Best epoch        : {ckpt.get('best_epoch', 'N/A')}")

    if 'best_val_acc' in ckpt:
        best_val_acc = ckpt['best_val_acc']
    elif 'history' in ckpt and 'val_acc' in ckpt['history']:
        best_val_acc = max(ckpt['history']['val_acc'])
    else:
        best_val_acc = None

    if best_val_acc is not None:
        print(f"Best val accuracy : {best_val_acc:.4f}")
    else:
        print("Best val accuracy : tidak ditemukan di checkpoint")

    if 'img_size' in ckpt:
        print(f"Image size        : {ckpt['img_size']}")
    if 'arch' in ckpt:
        print(f"Arsitektur        : {ckpt['arch']}")
    if 'model_name' in ckpt:
        print(f"Model name        : {ckpt['model_name']}")
    if 'best_params' in ckpt:
        print(f"Best params       : {ckpt['best_params']}")


def inspect_ensemble(ckpt: dict, path: Path):
    """Checkpoint gabungan (ensemble_best.pth) — skema key beda dari model tunggal."""
    state_dict = ckpt.get('state_dict', {})
    print(f"Jumlah tensor     : {len(state_dict)} (gabungan seluruh sub-model)")
    total_params = sum(v.numel() for v in state_dict.values())
    print(f"Total parameter   : {total_params / 1e6:.2f} M")

    print(f"Model keys        : {ckpt.get('model_keys', 'N/A')}")
    print(f"Model labels      : {ckpt.get('model_labels', 'N/A')}")
    print(f"Bobot ensemble    : {ckpt.get('weights', 'N/A')}")
    print(f"Image size (dict) : {ckpt.get('img_size', 'N/A')}")
    print(f"Formula objektif  : {ckpt.get('formula', 'N/A')}")
    print(f"Trial sumber      : {ckpt.get('source_trial', 'N/A')}")
    print(f"Num classes       : {ckpt.get('num_classes', 'N/A')}")
    print(f"ViT model name    : {ckpt.get('vit_model_name', 'N/A')}")

    if state_dict and 'model_keys' in ckpt:
        print("\nBreakdown parameter per sub-model:")
        for key in ckpt['model_keys']:
            prefix = f'models.{key}.'
            n_params = sum(v.numel() for k, v in state_dict.items() if k.startswith(prefix))
            n_tensor = sum(1 for k in state_dict if k.startswith(prefix))
            print(f"  {key:<14}: {n_tensor:>4} tensor, {n_params / 1e6:.2f} M params")


def inspect_checkpoint(path: Path):
    if not path.exists():
        print(f"[!] File tidak ditemukan: {path}\n")
        return

    ckpt = torch.load(path, map_location='cpu', weights_only=False)

    print(f"File              : {path}")
    print(f"Ukuran file       : {path.stat().st_size / 1e6:.2f} MB")
    print(f"Keys di dalam pth : {list(ckpt.keys())}")

    is_ensemble = 'weights' in ckpt and 'model_keys' in ckpt
    if is_ensemble:
        inspect_ensemble(ckpt, path)
    else:
        inspect_single_model(ckpt, path)

    print("-" * 70)


if __name__ == '__main__':
    for fname in CHECKPOINTS:
        inspect_checkpoint(CKPT_DIR / fname)

File              : D:\Tree-Structured Parzen Estimator\main\[6] model\densenet121_best.pth
Ukuran file       : 30.57 MB
Keys di dalam pth : ['arch', 'state_dict', 'best_params', 'best_epoch', 'history', 'img_size']
Jumlah tensor     : 734
Total parameter   : 7.57 M
Best epoch        : 31
Best val accuracy : 0.9214
Image size        : 224
Arsitektur        : densenet121
Best params       : {'lr': 0.004004743782924012, 'batch_size': 64, 'dropout': 0.36548432597632635, 'weight_decay': 1.3551027943175312e-06, 'optimizer_name': 'adamw', 'use_focal': False, 'focal_gamma': 1.7441316853461215, 'label_smoothing': 0.028325119630524025}
----------------------------------------------------------------------
File              : D:\Tree-Structured Parzen Estimator\main\[6] model\inception_v3_best.pth
Ukuran file       : 91.72 MB
Keys di dalam pth : ['arch', 'state_dict', 'best_params', 'best_epoch', 'history', 'img_size']
Jumlah tensor     : 573
Total parameter   : 22.87 M
Best epoch        : 32
Be